# Stata Analysis Code Template

This is a template for Stata analysis code. It is designed to be used in a Jupyter notebook.

* Key rules: 
    - EVERY cell should be self-contained and executable, so all global settings should be repeated in every cell.
    - Each cell usually write one table, panel or figure.
    - Do not write custom `program define` wrappers, Mata code, or meta/diagnostic machinery (postfiles, model-metadata logging, coefficient dumps); call estimators directly and store results with `est store` (plus `estadd` for table rows).
    - Keep the codes as few lines as possible, minimize unnecessary code beyond shown in the examples below.

* Output rules (always follow):
    - Tables: export to four formats with the same basename — `.md`, `.rtf`, `.csv`, `.tex` — using the `esttab` options shown below (`cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label`; add `r2_p` for nonlinear models).
    - Figures: export to both `.pdf` and `.png`.
    - No extra side files (`.ster`, metadata, diagnostic dumps); the four formats are the deliverable.

* Entry file: one `reg.dta` file per sample (one for a project that uses only one baseline sample)

## Example: baseline models

In [1]:
version 18
set more off

global fe "forecast_error"
global ar "abnormal_return"
global growth "forecast_growth"

use "data/equity_offerings/reg.dta", clear

est clear
mlexp (lnnormalden($fe, {a0=0} + {a1=0}*$growth, exp({lnsfe=0})) ///
       + lnnormalden($ar, {b1=1}*($fe - {a0star=0} - {a1star=0}*$growth), ///
                     exp({lnsar=0}))), variables($fe $ar $growth)
estimates store unrestricted

mlexp (lnnormalden($fe, {a0=0} + {a1=0}*$growth, exp({lnsfe=0})) ///
       + lnnormalden($ar, {b1=1}*($fe - {a0} - {a1}*$growth), ///
                     exp({lnsar=0}))), variables($fe $ar $growth)
estimates store market_efficiency
lrtest unrestricted market_efficiency
local lr_me_chi2 = r(chi2)
local lr_me_p = r(p)

mlexp (lnnormalden($fe, {a0=0} + {a1=0}*$growth, exp({lnsfe=0})) ///
       + lnnormalden($ar, {b1=1}*$fe, exp({lnsar=0}))), variables($fe $ar $growth)
estimates store naive_expectations
lrtest unrestricted naive_expectations
local lr_naive_chi2 = r(chi2)
local lr_naive_p = r(p)

estimates restore unrestricted
regress $fe $growth
estimates restore unrestricted
nlcom (ols_intercept: -_b[/b1]*_b[/a0star]) ///
      (ols_growth_slope: -_b[/b1]*_b[/a1star]) ///
      (ols_forecast_error_slope: _b[/b1])
regress $ar $fe $growth
estimates restore unrestricted
estadd scalar lr_me_chi2 = `lr_me_chi2'
estadd scalar lr_me_p = `lr_me_p'
estadd scalar lr_naive_chi2 = `lr_naive_chi2'
estadd scalar lr_naive_p = `lr_naive_p'

esttab unrestricted using "tables/table7_joint_system.md", ///
    cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) ///
    keep(a0:_cons a1:_cons b1:_cons a0star:_cons a1star:_cons) ///
    stats(N ll lr_me_chi2 lr_me_p lr_naive_chi2 lr_naive_p, ///
    labels("Observations" "Log likelihood" "Market-efficiency LR chi-square" ///
           "Market-efficiency LR p-value" "Naive-expectations LR chi-square" ///
           "Naive-expectations LR p-value") fmt(0 3 3 3 3 3)) replace label
esttab unrestricted using "tables/table7_joint_system.rtf", ///
    cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) ///
    keep(a0:_cons a1:_cons b1:_cons a0star:_cons a1star:_cons) ///
    stats(N ll lr_me_chi2 lr_me_p lr_naive_chi2 lr_naive_p, ///
    labels("Observations" "Log likelihood" "Market-efficiency LR chi-square" ///
           "Market-efficiency LR p-value" "Naive-expectations LR chi-square" ///
           "Naive-expectations LR p-value") fmt(0 3 3 3 3 3)) replace label
esttab unrestricted using "tables/table7_joint_system.csv", ///
    cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) ///
    keep(a0:_cons a1:_cons b1:_cons a0star:_cons a1star:_cons) ///
    stats(N ll lr_me_chi2 lr_me_p lr_naive_chi2 lr_naive_p, ///
    labels("Observations" "Log likelihood" "Market-efficiency LR chi-square" ///
           "Market-efficiency LR p-value" "Naive-expectations LR chi-square" ///
           "Naive-expectations LR p-value") fmt(0 3 3 3 3 3)) replace label
esttab unrestricted using "tables/table7_joint_system.tex", ///
    cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) ///
    keep(a0:_cons a1:_cons b1:_cons a0star:_cons a1star:_cons) ///
    stats(N ll lr_me_chi2 lr_me_p lr_naive_chi2 lr_naive_p, ///
    labels("Observations" "Log likelihood" "Market-efficiency LR chi-square" ///
           "Market-efficiency LR p-value" "Naive-expectations LR chi-square" ///
           "Naive-expectations LR p-value") fmt(0 3 3 3 3 3)) replace label
est clear



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 16-core network, expiring 15 Dec 2024
Serial number: 501809303220
  Licensed to: Luther
               Users

Notes:
      1. Stata is running in batch mode.
      2. Unicode is supported; see help unicode_advice.
      3. More than 2 billion observations are allowed; see help obs_advice.
      4. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Running /usr/local/stata18/sysprofile.do ...

. do

## Example: descriptive statistics

In [1]:
version 18
set more off

use "data/equity_offerings/reg.dta", clear

est clear
estpost tabstat abnormal_return realized_growth forecast_growth forecast_error, ///
    statistics(n mean sd p25 p50 p75) columns(statistics)
esttab using "tables/table7_descriptive.md", replace ///
    cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") label
esttab using "tables/table7_descriptive.rtf", replace ///
    cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") label
esttab using "tables/table7_descriptive.csv", replace ///
    cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") label
esttab using "tables/table7_descriptive.tex", replace ///
    cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") label
est clear



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 16-core network, expiring 15 Dec 2024
Serial number: 501809303220
  Licensed to: Luther
               Users

Notes:
      1. Stata is running in batch mode.
      2. Unicode is supported; see help unicode_advice.
      3. More than 2 billion observations are allowed; see help obs_advice.
      4. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Running /usr/local/stata18/sysprofile.do ...

. do